# Mi Primera Red Neuronal — laboratorio progresivo

**Objetivo:** comprender matrices, implementaciones de neuronas, forward pass, pérdida, backpropagation y evaluación; finalmente publicar una aplicación con cámara. 

Clase **0 = triste**; clase **1 = feliz**. *Dibujos*, no emociones de personas.

**Bibliografía científica principal:** [Rosenblatt (1958)](https://doi.org/10.1037/h0042519) · [Rumelhart, Hinton & Williams (1986)](https://doi.org/10.1038/323533a0) · [Glorot & Bengio (2010)](https://proceedings.mlr.press/v9/glorot10a.html).

## 0. Cómo ejecutarlo

Abre Jupyter desde la **raíz del repositorio**:

```bash
pip install -r requirements.txt
jupyter notebook
```

Este notebook importa el código abierto del propio repositorio. No necesitas conexión durante el entrenamiento: el dataset **está incluido** en `data/`.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'data/caritas_dataset.npz').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print('Proyecto:', ROOT)
import numpy as np
import matplotlib.pyplot as plt
from src.preprocess import load_datasets, preprocess_image
from src.neural_network import sigmoid, relu, MLP, gradient_check
from src.train import accuracy

## 1. ¿Cómo ve una computadora una imagen?

Una carita en escala de grises se convierte en una matriz de **28 × 28**. Para nuestra red, cada imagen pasa a un vector de **784** valores de intensidad de tinta en el intervalo **[0,1]**.

In [ ]:
d = load_datasets(str(ROOT / 'data/caritas_dataset.npz'))
for key,value in d.items(): print(key, value.shape)
X_train, y_train = d['X_train'], d['y_train']
X_val, y_val = d['X_val'], d['y_val']
X_test, y_test = d['X_test'], d['y_test']
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(X_train[i].reshape(28,28), cmap='gray',vmin=0,vmax=1)
    ax.set_title('Feliz' if int(y_train[i,0]) else 'Triste');ax.axis('off')
plt.tight_layout();plt.show()

## 2. Vectores y producto punto

Ejemplo de una neurona con tres entradas: `z = x @ w + b`.

Primero calcula a mano y después contrasta con NumPy.

In [ ]:
x = np.array([0.2, 0.8, 0.5])
w = np.array([0.4, -0.3, 0.9])
b = 0.1
z = x @ w + b
print('Suma ponderada:', z)
assert np.isclose(z, 0.39)  # 0.08 - 0.24 + 0.45 + 0.1

## 3. Activaciones: no linealidad

`ReLU(z) = max(0,z)` y `sigmoid(z)=1/(1+exp(-z))`. ReLU activa las neuronas ocultas y sigmoide produce una salida entre 0 y 1.

In [ ]:
z_values = np.linspace(-6,6,300)
plt.plot(z_values,relu(z_values),label='ReLU')
plt.plot(z_values,sigmoid(z_values),label='Sigmoide')
plt.grid(alpha=.2);plt.legend();plt.title('Funciones de activación');plt.show()

## 4. Arquitectura matricial 784 → 32 → 16 → 1

Usamos ejemplos por filas (`batch_size × 784`). Para cada capa `Z = A @ W + b`.

| Tensor | Forma |
|---|---|
| X | `(B, 784)` |
| W1 / b1 | `(784, 32)` / `(1, 32)` |
| W2 / b2 | `(32, 16)` / `(1, 16)` |
| W3 / b3 | `(16, 1)` / `(1, 1)` |

In [ ]:
rng = np.random.default_rng(42)
W1 = (rng.standard_normal((784,32))*np.sqrt(2/784)).astype(np.float32)
b1 = np.zeros((1,32), dtype=np.float32)
W2 = (rng.standard_normal((32,16))*np.sqrt(2/32)).astype(np.float32)
b2 = np.zeros((1,16), dtype=np.float32)
W3 = (rng.standard_normal((16,1))*np.sqrt(1/16)).astype(np.float32)
b3 = np.zeros((1,1), dtype=np.float32)
print('W1, W2, W3:', W1.shape, W2.shape, W3.shape)

## 5. Forward propagation, paso a paso

Una fila del batch pasa por todas las capas. El resultado final `p` es una puntuación asociada a la clase feliz.

In [ ]:
def forward_manual(X):
    Z1 = X @ W1 + b1; A1 = relu(Z1)
    Z2 = A1 @ W2 + b2; A2 = relu(Z2)
    Z3 = A2 @ W3 + b3; P = sigmoid(Z3)
    cache = (X, Z1, A1, Z2, A2, Z3, P)
    return P, cache
probs, cache = forward_manual(X_train[:4])
print('Probabilidades iniciales:', probs.ravel())

## 6. ¿Cómo medimos el error?

Usamos **Binary Cross Entropy (BCE)**. Para etiquetas `y` binarias y salida `p`:

`L = -mean(y*log(p) + (1-y)*log(1-p))`.

Computamos la versión numéricamente estable directamente con los logits `z3`: `mean(logaddexp(0,z3) - y*z3)`.

In [ ]:
def bce_from_logits(z3, y):
    return np.mean(np.logaddexp(0,z3) - y*z3)
print('Pérdida inicial:', bce_from_logits(cache[5], y_train[:4]))

## 7. Backpropagation: la regla de la cadena

Para salida `sigmoide + BCE`, `dZ3=(P-y)/B`. Los gradientes de una capa son `dW=A_prev.T @ dZ`, `db=sum(dZ)`. Luego el gradiente vuelve a la capa anterior y se multiplica por la derivada de ReLU.

In [ ]:
def gradients_manual(X, y):
    P, (X,Z1,A1,Z2,A2,Z3,P) = forward_manual(X)
    B = len(X)
    dZ3 = (P-y)/B
    dW3 = A2.T @ dZ3; db3 = dZ3.sum(axis=0,keepdims=True)
    dZ2 = (dZ3 @ W3.T)*(Z2>0)
    dW2 = A1.T @ dZ2; db2 = dZ2.sum(axis=0,keepdims=True)
    dZ1 = (dZ2 @ W2.T)*(Z1>0)
    dW1 = X.T @ dZ1; db1 = dZ1.sum(axis=0,keepdims=True)
    return (dW1,db1,dW2,db2,dW3,db3)
grads = gradients_manual(X_train[:8], y_train[:8])
print('Gradientes:', [g.shape for g in grads])

## 8. Comprobación numérica del gradiente

Una derivada calculada por la regla de la cadena debería parecerse a `(L(θ+ε)-L(θ-ε))/(2ε)`.

Probamos sobre una red pequeña en doble precisión, antes del entrenamiento completo.

In [ ]:
tiny = MLP(layers=(4,5,3,1), seed=20, dtype=np.float64)
Xs = np.random.default_rng(3).random((5,4))
ys = np.array([0,1,0,1,1]).reshape(-1,1)
print('Error relativo máximo:', gradient_check(tiny,Xs,ys,epsilon=1e-5,checks=30))

## 9. Nuestro primer entrenamiento, sin frameworks

Descenso de gradiente mini-batch sobre los pesos y sesgos definidos arriba. Para acelerar el ejercicio usamos menos épocas que el modelo previamente exportado.

In [ ]:
learning_rate = 0.25
history=[]
shuffle = np.random.default_rng(43)
for epoch in range(20):
    idx = shuffle.permutation(len(X_train))
    for ids in np.array_split(idx, int(np.ceil(len(idx)/64))):
        x,y = X_train[ids],y_train[ids]
        dW1,db1,dW2,db2,dW3,db3 = gradients_manual(x,y)
        W1 -= learning_rate*dW1; b1 -= learning_rate*db1
        W2 -= learning_rate*dW2; b2 -= learning_rate*db2
        W3 -= learning_rate*dW3; b3 -= learning_rate*db3
    val_p,val_cache=forward_manual(X_val)
    val_loss=float(bce_from_logits(val_cache[5],y_val))
    val_acc=float(np.mean((val_p.ravel()>=0.5)==y_val.ravel()))
    history.append((val_loss,val_acc))
    if (epoch+1)%5==0:print(f'Época {epoch+1:02d} | loss val={val_loss:.4f} | acc val={val_acc:.3f}')

## 10. Curvas de aprendizaje

Las curvas permiten observar si la pérdida disminuye y si el rendimiento de validación se estabiliza. Ajusta hiperparámetros usando **validación**, no test.

In [ ]:
plt.plot([x[0] for x in history]);plt.title('Pérdida de validación');plt.xlabel('Época');plt.ylabel('BCE');plt.grid(alpha=.2);plt.show()

## 11. Test reservado y matriz de confusión

Evaluamos una sola vez los pesos elegidos usando validación. El modelo incluido en el repositorio se entrenó con `python -m src.train` y ofrece una referencia reproducible para el taller.

In [ ]:
from src.train import load_model
packaged = load_model(str(ROOT/'models/weights.npz'))
y_true = y_test.ravel().astype(int)
y_pred = packaged.predict(X_test)
cm = np.zeros((2,2),dtype=int)
for real,pred in zip(y_true,y_pred):cm[real,pred]+=1
print('Accuracy test SINTÉTICO:', (y_true==y_pred).mean())
print('Matriz: filas real, columnas predicho:', cm)
print('No implica la misma precisión en papel o fotos reales.')

## 12. Exportar el mismo modelo a la aplicación de cámara

`demo/model.json` contiene **las matrices realmente entrenadas**; la aplicación calcula la inferencia con multiplicaciones matriciales equivalentes en JavaScript.

In [ ]:
from src.export_model import export
# El modelo original empaquetado:
model_file = ROOT / 'demo/model.json'
print('Modelo listo:', model_file, 'bytes:', model_file.stat().st_size)
# Para exportar tu nuevo entrenamiento desde el CLI:
print('python -m src.train && python -m src.export_model')

## 13. ¿Cómo comprobar con dibujos reales?

1. Abre `demo/` desde GitHub Pages en un teléfono.
2. Dibuja una carita feliz o triste sobre una hoja blanca.
3. Encuádralo completamente, evita sombras y toma la captura.
4. Guarda una tabla propia de aciertos y errores. Prueba trazos variados.
5. Añade dibujos reales **al entrenamiento** solo después de definir un test real independiente.

**Importante:** no uses este clasificador para inferir emociones de personas reales.

## 14. Retos para practicar

- Cambia el número de neuronas ocultas y compara loss/tiempo.
- Varía `learning_rate` entre 0.05 y 0.5.
- Mide qué sucede si entrenas con menos datos.
- Cambia la semilla y analiza la variabilidad.
- Fotografía al menos 20 dibujos nuevos y reporta los errores.
- Investiga por qué una CNN puede ser mejor para imágenes 2D.

**Referencias:** [Rumelhart et al. (1986)](https://doi.org/10.1038/323533a0), [Glorot & Bengio (2010)](https://proceedings.mlr.press/v9/glorot10a.html), [LeCun et al. (1998)](https://doi.org/10.1109/5.726791).